# 03.4 Transformer Encoder Intro / Transformer Encoder 入门

`Transformer Encoder` 可以理解成把前一节的 `self-attention` 放进一个更完整、更可训练的模块里。  
A `Transformer Encoder` can be understood as a more complete and trainable module that places `self-attention` into a larger structure.

这一节的重点不是背 API，而是建立结构直觉 / The goal here is not to memorize APIs, but to build structural intuition:

- 输入 token 如何变成向量 / how input tokens become vectors
- 为什么需要位置编码 / why positional encoding is needed
- 一个 encoder block 里面到底发生了什么 / what actually happens inside an encoder block
- shape 在各层之间怎么流动 / how shapes flow across the layers

## 学习目标 / Learning Goals

学完后你应该能 / After this notebook, you should be able to:

1. 说清 `Transformer Encoder` 的基本组成 / Explain the main components of a `Transformer Encoder`.
2. 理解 `token embedding / positional encoding / self-attention / feed-forward network` 的角色 / Understand the roles of `token embedding / positional encoding / self-attention / feed-forward network`.
3. 跟踪 `batch_size / seq_len / d_model` 的 shape 变化 / Track the shape flow of `batch_size / seq_len / d_model`.
4. 理解 `residual connection / layer normalization` 的位置 / Understand where `residual connection / layer normalization` are used.
5. 用 `PyTorch` 写一个最小 encoder block / Write a minimal encoder block in `PyTorch`.
6. 训练一个极小的 Transformer 分类器做 toy task / Train a tiny Transformer classifier on a toy task.

In [ ]:
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

## 1. Encoder 的整体结构 / The Overall Encoder Structure

先记住一个够实用的流程图 / Start with this practical flow:

1. `token ids -> token embedding`
2. `token embedding + positional encoding`
3. `multi-head self-attention`
4. `residual connection + layer normalization`
5. `feed-forward network`
6. `residual connection + layer normalization`

一个 encoder 往往会堆叠多个这样的 block。  
An encoder usually stacks multiple blocks like this.

In [ ]:
token_ids = torch.tensor(
    [
        [2, 5, 7, 0, 0],
        [4, 6, 3, 8, 9],
    ],
    dtype=torch.long,
)

vocab_size = 12
d_model = 16
embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
x = embedding(token_ids)

print("token_ids.shape =", token_ids.shape)
print("x.shape after embedding =", x.shape)

这里的 shape 是：  
The shape here is:

- `token_ids.shape == (batch_size, seq_len)`
- `embedding_output.shape == (batch_size, seq_len, d_model)`

`d_model` 就是每个 token 最终对应的表示维度 / `d_model` is the representation dimension of each token.

## 2. Positional Encoding / 位置编码

`Self-Attention` 本身并不天然知道“第一个 token”“最后一个 token”这些位置信息。  
`Self-Attention` by itself does not naturally know which token is first or last.

所以我们需要把位置注入输入表示 / So we need to inject position into the input representation.

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=128):
        super().__init__()
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]


pos_encoder = SinusoidalPositionalEncoding(d_model=d_model, max_len=32)
x_with_pos = pos_encoder(x)

print("x_with_pos.shape =", x_with_pos.shape)
print("position 0 encoding slice =", pos_encoder.pe[0, 0, :6])
print("position 1 encoding slice =", pos_encoder.pe[0, 1, :6])

位置编码 / positional encoding 的重点不是把公式背下来，而是记住它的功能：  
The key point of positional encoding is not memorizing the formula, but remembering its function:

- 同样的 token，放在不同位置时，表示应该不完全一样 / the same token at different positions should not have exactly the same representation

## 3. Multi-Head Self-Attention / 多头自注意力

`Multi-Head Attention` 的直觉是：  
The intuition of `Multi-Head Attention` is:

- 不是只学一套注意力模式 / do not learn only one attention pattern
- 而是并行学习多套注意力模式 / learn multiple attention patterns in parallel

In [ ]:
key_padding_mask = token_ids == 0
mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=2, batch_first=True, dropout=0.0)

attn_output, attn_weights = mha(
    x_with_pos,
    x_with_pos,
    x_with_pos,
    key_padding_mask=key_padding_mask,
    need_weights=True,
    average_attn_weights=False,
)

print("key_padding_mask =\n", key_padding_mask)
print("attn_output.shape =", attn_output.shape)
print("attn_weights.shape =", attn_weights.shape)

这里的 shape 非常关键 / These shapes are very important:

- `attn_output.shape == (batch_size, seq_len, d_model)`
- `attn_weights.shape == (batch_size, num_heads, seq_len, seq_len)`

最后一个 `seq_len` 表示“每个位置要对整条序列分配权重”。  
The final `seq_len` means each position assigns weights across the whole sequence.

## 4. 一个最小 Encoder Block / A Minimal Encoder Block

下面把 attention、残差连接 / residual connection、层归一化 / layer normalization、前馈网络 / feed-forward network 拼起来。  
Now we combine attention, residual connection, layer normalization, and feed-forward network into one block.

In [ ]:
class SimpleTransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, nhead, ff_hidden):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, batch_first=True, dropout=0.0)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_hidden),
            nn.ReLU(),
            nn.Linear(ff_hidden, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, key_padding_mask=None):
        attn_out, attn_weights = self.attn(
            x,
            x,
            x,
            key_padding_mask=key_padding_mask,
            need_weights=True,
            average_attn_weights=False,
        )
        x = self.norm1(x + attn_out)
        ff_out = self.ffn(x)
        x = self.norm2(x + ff_out)
        return x, attn_weights


block = SimpleTransformerEncoderBlock(d_model=d_model, nhead=2, ff_hidden=32)
block_output, block_attn = block(x_with_pos, key_padding_mask=key_padding_mask)

print("block_output.shape =", block_output.shape)
print("block_attn.shape =", block_attn.shape)

注意：encoder block 不会改变 `(batch_size, seq_len, d_model)` 这个主 shape。  
Notice that an encoder block does not change the main `(batch_size, seq_len, d_model)` shape.

这也是深层堆叠变得方便的原因之一 / This is one reason why deep stacking is convenient.

## 5. 从 Encoder 到分类器 / From Encoder to a Classifier

做序列分类时，常见做法之一是：  
For sequence classification, one common strategy is:

1. 把整条序列编码成隐藏表示 / encode the whole sequence into hidden states
2. 做 pooling（例如 mean pooling）/ apply pooling such as mean pooling
3. 接一个线性分类头 / attach a linear classification head

In [ ]:
def masked_mean(x, mask):
    valid = (~mask).unsqueeze(-1).float()
    summed = (x * valid).sum(dim=1)
    counts = valid.sum(dim=1).clamp(min=1.0)
    return summed / counts


class TinyTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=16, nhead=2, ff_hidden=32, num_layers=1, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_encoder = SinusoidalPositionalEncoding(d_model=d_model, max_len=32)
        self.blocks = nn.ModuleList(
            [SimpleTransformerEncoderBlock(d_model, nhead, ff_hidden) for _ in range(num_layers)]
        )
        self.head = nn.Linear(d_model, 2)

    def forward(self, input_ids):
        key_padding_mask = input_ids == self.pad_id
        x = self.embedding(input_ids)
        x = self.pos_encoder(x)

        attn_maps = []
        for block in self.blocks:
            x, attn_weights = block(x, key_padding_mask=key_padding_mask)
            attn_maps.append(attn_weights)

        pooled = masked_mean(x, key_padding_mask)
        logits = self.head(pooled)
        return logits, attn_maps

## 6. 一个 Toy Task / A Toy Task

为了让 notebook 保持可跑且足够直观，这里造一个非常小的任务：  
To keep the notebook runnable and intuitive, we build a very small task:

- 输入是变长 token 序列 / the input is a variable-length token sequence
- 标签由“第一个 token 是否为偶数”决定 / the label is determined by whether the first token is even

这个任务的意义在于 / The point of this task is:

- 模型必须区分“第一个位置” / the model must distinguish the first position
- 所以位置编码 / positional encoding 就是必要的

In [ ]:
def make_dataset(n_samples, max_len=6, vocab_low=1, vocab_high=10):
    xs = []
    ys = []
    for _ in range(n_samples):
        actual_len = torch.randint(3, max_len + 1, (1,)).item()
        tokens = torch.randint(vocab_low, vocab_high + 1, (actual_len,)).tolist()
        label = int(tokens[0] % 2 == 0)
        padded = tokens + [0] * (max_len - actual_len)
        xs.append(padded)
        ys.append(label)
    return torch.tensor(xs, dtype=torch.long), torch.tensor(ys, dtype=torch.long)


X_train, y_train = make_dataset(480)
X_val, y_val = make_dataset(120)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=64, shuffle=False)

print("X_train.shape =", X_train.shape)
print("y_train.shape =", y_train.shape)
print("positive rate / 正样本比例 =", y_train.float().mean().item())

In [ ]:
def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_items = 0

    for xb, yb in loader:
        with torch.set_grad_enabled(is_train):
            logits, _ = model(xb)
            loss = loss_fn(logits, yb)

        if is_train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        preds = logits.argmax(dim=1)
        total_loss += loss.item() * xb.size(0)
        total_correct += (preds == yb).sum().item()
        total_items += xb.size(0)

    return total_loss / total_items, total_correct / total_items


model = TinyTransformerClassifier(vocab_size=12, d_model=16, nhead=2, ff_hidden=32, num_layers=1)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

history = []
for epoch in range(1, 7):
    train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)
    history.append((epoch, train_loss, train_acc, val_loss, val_acc))
    print(
        f"epoch={epoch:02d} | train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
    )

In [ ]:
sample_x = X_val[:5]
sample_y = y_val[:5]
logits, attn_maps = model(sample_x)
preds = logits.argmax(dim=1)

print("sample_x =\n", sample_x)
print("true labels =", sample_y)
print("pred labels =", preds)
print("last attention map shape =", attn_maps[-1].shape)

In [ ]:
# 练习 1 / Exercise 1
# 已知输入 x.shape == (4, 7, 16)，并使用 2 个 attention heads。
# Given x.shape == (4, 7, 16) and we use 2 attention heads.
#
# 一个 encoder block 的输出 shape 是多少？
# What is the output shape of one encoder block?
#
# 如果设置 average_attn_weights=False，attn_weights.shape 又是多少？
# If average_attn_weights=False, what is attn_weights.shape?

练习 1 参考答案 / Exercise 1 Reference Answer

- `output.shape == (4, 7, 16)`
- `attn_weights.shape == (4, 2, 7, 7)`

因为 encoder block 不改变主表示维度，只是在内部重新混合信息。  
Because the encoder block does not change the main representation shape; it only remixes information internally.

In [ ]:
# 练习 2 / Exercise 2
# 为什么没有位置编码 / positional encoding 时，
# Why is it hard for a Transformer encoder without positional encoding
# 配合 mean pooling 去区分 [2, 7, 5] 和 [5, 7, 2] 这种只有顺序不同的输入？
# to distinguish inputs like [2, 7, 5] and [5, 7, 2] that differ only by order?

练习 2 参考答案 / Exercise 2 Reference Answer

因为如果没有位置编码，模型看到的更像是“一个 token 集合 / a set of tokens”，而不是有顺序的序列。  
Without positional encoding, the model sees something closer to "a set of tokens" than an ordered sequence.

再加上 `mean pooling` 会进一步压掉顺序差异，所以模型很难知道谁在第一个位置。  
And `mean pooling` further removes order differences, so the model has a hard time knowing which token was in the first position.

## 7. 小结 / Summary

这一节最重要的收获 / The most important takeaways from this notebook are:

1. `Transformer Encoder` 的基本结构是：embedding + positional encoding + attention + FFN / The basic encoder structure is embedding + positional encoding + attention + FFN.
2. `self-attention` 负责跨位置聚合信息 / `self-attention` aggregates information across positions.
3. `positional encoding` 负责告诉模型“顺序” / `positional encoding` tells the model about order.
4. encoder block 通常保持 `(batch_size, seq_len, d_model)` 不变 / an encoder block usually keeps `(batch_size, seq_len, d_model)` unchanged.
5. 做分类时，可以把 encoder 输出做 pooling 再接线性头 / for classification, we can pool encoder outputs and attach a linear head.

下一步建议 / Suggested next step:

- 如果你想接触更贴近真实 NLP 工作流的内容，可以继续看“预训练文本模型 / pretrained text models”的可选 notebook。  
- If you want something closer to real NLP workflows, the next natural step is the optional notebook on pretrained text models.